In [1]:
pip install chromadb pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.6 MB/s eta

In [2]:
# ==============================================================================
# SETUP: LOAD EXAMPLES INTO CHROMADB (Run Once)
# ==============================================================================
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

# 1. Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db_examples")
ef = embedding_functions.DefaultEmbeddingFunction()
example_collection = chroma_client.get_or_create_collection(name="training_examples", embedding_function=ef)

# 2. Load your CSV
# Ensure your CSV has these columns: Requirement, Actor, Goal, Rationale, FR_NFR, NFR subtype
try:
    df_examples = pd.read_csv('/content/parsed_requirements.csv') # <--- CHANGE FILENAME IF NEEDED
    df_examples = df_examples.fillna('') # Handle empty fields

    ids = []
    documents = []
    metadatas = []

    print(f"🔄 Loading {len(df_examples)} examples into Vector Database...")

    for index, row in df_examples.iterrows():
        ids.append(f"ex_{index}")
        documents.append(row['Requirement'])

        # Store the answers in metadata so we can inject them into the prompt
        metadatas.append({
            "actor": row['Actor'],
            "goal": row['Goal'],
            "rationale": row['Rationale'],
            "type": row['FR_NFR'],
            "subtype": row['Subtype']
        })

    # Add to ChromaDB
    example_collection.add(ids=ids, documents=documents, metadatas=metadatas)
    print("✅ Examples loaded. Dynamic Few-Shot retrieval is ready.")

except FileNotFoundError:
    print("❌ 'requirements.csv' not found. Please upload your training data CSV.")

🔄 Loading 1007 examples into Vector Database...


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 104MiB/s]


✅ Examples loaded. Dynamic Few-Shot retrieval is ready.


In [3]:
def get_dynamic_examples(input_requirement, mode="extraction"):
    """
    Retrieves top 3 similar examples from ChromaDB and formats them for the prompt.
    Modes: 'extraction' or 'classification'
    """
    results = example_collection.query(
        query_texts=[input_requirement],
        n_results=3
    )

    formatted_examples = ""

    for i in range(len(results['ids'][0])):
        doc = results['documents'][0][i]
        meta = results['metadatas'][0][i]

        if mode == "extraction":
            # Format for Phase 2
            formatted_examples += f"""
--- EXAMPLE {i+1} ---
Requirement: "{doc}"
{{
  "actor": "{meta['actor']}",
  "goal": "{meta['goal']}",
  "rationale": "{meta['rationale']}"
}}
"""
        elif mode == "classification":
            # Format for Phase 3
            formatted_examples += f"""
--- EXAMPLE {i+1} ---
Requirement: "{doc}"
{{
  "type": "{meta['type']}",
  "subtype": "{meta['subtype']}",
  "reasoning": "Based on similar past examples."
}}
"""
    return formatted_examples

In [7]:
# ==============================================================================
# GAI-Enhanced Requirements Engineering Management Process (REMP) System
# Thesis Demonstration Notebook
# ==============================================================================


# ==============================================================================
# LIBRARIES & SETUP
# ==============================================================================
import os
import re
import time
import json
import random
import logging
from uuid import uuid4
from tqdm.auto import tqdm

# --- Install necessary libraries ---
!pip install -q -U google-generativeai tqdm transformers sentence-transformers

# --- Configure logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Import necessary components ---
import google.generativeai as genai
from google.colab import userdata

# ==============================================================================
# CONFIGURATION & GLOBAL SETTINGS
# ==============================================================================
class Config:
    try:
        GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
        if not GEMINI_API_KEY:
            raise ValueError("API Key not found.")
        genai.configure(api_key=GEMINI_API_KEY)
        logging.info("Gemini API Key loaded successfully from Colab Secrets.")
    except Exception as e:
        GEMINI_API_KEY = None
        logging.warning("Could not load Gemini API Key. GAI features will be disabled.")

    GAI_MODEL_NAME = "gemini-2.5-pro"
    USE_REAL_MODELS = False
    USE_GAI_PHASE1 = True
    USE_GAI_PHASE2 = True
    USE_GAI_PHASE3 = True
    USE_GAI_PHASE4 = False
    USE_GAI_PHASE6 = False
    CONFIDENCE_THRESHOLD = 0.85
    QUALITY_SCORE_THRESHOLD = 0.85
    API_CALL_DELAY_SECONDS = 3
    MAX_API_RETRIES = 3
    BACKOFF_FACTOR = 2
    VALIDATION_THRESHOLD = 0.80 # Added for Phase 4 validation

class ApiManager:
    def __init__(self):
        self.call_count = 0
        self.last_call_time = 0

    def make_api_call(self, prompt, is_json_output=False):
        if not Config.GEMINI_API_KEY:
            logging.error("GAI is unavailable. No API Key configured.")
            return None

        elapsed_time = time.time() - self.last_call_time
        if elapsed_time < Config.API_CALL_DELAY_SECONDS:
            time.sleep(Config.API_CALL_DELAY_SECONDS - elapsed_time)

        for attempt in range(Config.MAX_API_RETRIES):
            try:
                self.last_call_time = time.time()
                model = genai.GenerativeModel(Config.GAI_MODEL_NAME)
                generation_config = {"response_mime_type": "application/json"} if is_json_output else {}
                response = model.generate_content(prompt, generation_config=generation_config)
                self.call_count += 1
                logging.info(f"API call successful. Total calls: {self.call_count}")
                if self.call_count > 1000:
                    logging.warning(f"API call count is high ({self.call_count}). Monitor your daily quota.")
                return response.text
            except Exception as e:
                if "429" in str(e) or "quota" in str(e).lower():
                    wait_time = Config.API_CALL_DELAY_SECONDS * (Config.BACKOFF_FACTOR ** attempt)
                    logging.warning(f"Rate limit exceeded. Retrying in {wait_time:.2f}s... (Attempt {attempt + 1}/{Config.MAX_API_RETRIES})")
                    time.sleep(wait_time)
                else:
                    logging.error(f"An unexpected API error occurred: {e}")
                    return None
        logging.error("API call failed after multiple retries.")
        return None

api_manager = ApiManager()

# ==============================================================================
# PHASE 1: INPUT QUALITY CHECK
# ==============================================================================
def phase1_input_quality_check(requirement_text):
    print("\n--- PHASE 1: Input Quality Check ---")
    original_text = requirement_text

    actor_pattern = r'As an?|As the'
    goal_pattern = r'I want to|system shall|must|needs to|should be able to|the ability to|requires|I need to|application should|must support|feature will'
    rationale_pattern = r'so that|in order to|for the purpose of|because|to improve|to prevent|to ensure|which results in|as a result|due to|to achieve'

    has_actor = bool(re.search(actor_pattern, requirement_text, re.I))
    has_goal = bool(re.search(goal_pattern, requirement_text, re.I))
    has_rationale = bool(re.search(rationale_pattern, requirement_text, re.I))

    is_complete = has_actor and has_goal and has_rationale

    if is_complete:
        print("✅ Requirement appears complete. Skipping GAI enhancement.")
        return original_text

    print("⚠️ Requirement is incomplete. Missing parts identified.")
    if not Config.USE_GAI_PHASE1:
        print("GAI for Phase 1 is disabled. Proceeding with original text.")
        return original_text

    print("🤖 Engaging GAI to enhance the requirement (Few-Shot)...")
    prompt = f"""
    The following software requirement is incomplete. Enhance it by adding ONLY the missing components (Actor, Goal, or Rationale) to make it a complete user story in the format "As a [Actor], I want [Goal], so that [Rationale]".
    RULES:
    1. STRICTLY preserve the original meaning. Do not add new features.
    2. Keep the enhancement concise and realistic.
    3. The final text should not be more than double the length of the original.
    4. Output only the final, enhanced requirement text. No explanations.
    --- EXAMPLE ---
    Original Requirement: "User needs fast login"
    Enhanced Requirement: As a user, I want to log in quickly, so that I can access my account without waiting.
    --- END EXAMPLE ---
    Original Requirement: "{original_text}"
    Enhanced Requirement:
    """
    enhanced_text = api_manager.make_api_call(prompt)

    if enhanced_text:
        if len(enhanced_text) > (len(original_text) * 2.5) and len(original_text) > 10:
            print("❌ GAI enhancement is too long. Reverting to original text.")
            return original_text
        else:
            print(f"✨ GAI Enhanced Requirement: {enhanced_text.strip()}")
            return enhanced_text.strip()
    else:
        print("❌ GAI enhancement failed. Reverting to original text.")
        return original_text

# ==============================================================================
# PHASE 2: EXTRACTION (ACTOR, GOAL, RATIONALE)
# ==============================================================================
def phase2_extraction(requirement_text):
    print("\n--- PHASE 2: Extraction (with RAG) ---")

    # 1. Try Regex First (Fast/Cheap)
    actor_match = re.search(r'As a(?:n)?\s(.*?),\s*I want', requirement_text, re.I)
    goal_match = re.search(r'I want to\s(.*?)(?:,\s*so that|\s*so that)', requirement_text, re.I)
    rationale_match = re.search(r'so that\s(.*?)$', requirement_text, re.I)

    extracted = {
        "actor": actor_match.group(1).strip() if actor_match else None,
        "goal": goal_match.group(1).strip() if goal_match else None,
        "rationale": rationale_match.group(1).strip() if rationale_match else None
    }

    # Calculate basic confidence
    parts_found = sum(1 for part in extracted.values() if part is not None)
    confidence = parts_found / 3.0
    print(f"📊 Regex extraction confidence: {confidence:.2f}")

    # 2. If Confidence Low -> USE RETRIEVAL AUGMENTED GENERATION
    if confidence < Config.CONFIDENCE_THRESHOLD and Config.USE_GAI_PHASE2:
        print(f"🤖 Engaging GAI with Dynamic Retrieval (RAG)...")

        # A. RETRIEVE 3 SIMILAR EXAMPLES
        dynamic_few_shot = get_dynamic_examples(requirement_text, mode="extraction")

        # B. FORM AUGMENTED PROMPT
        prompt = f"""
        Analyze the software requirement and extract 'Actor', 'Goal', and 'Rationale'.
        Respond with a JSON object. Use the provided examples as a guide.

        {dynamic_few_shot}

        --- YOUR TASK ---
        Requirement: "{requirement_text}"
        """

        # C. GENERATE Y
        json_response = api_manager.make_api_call(prompt, is_json_output=True)

        if json_response:
            try:
                gai_extracted = json.loads(json_response)
                print("✨ GAI extraction complete (Augmented with DB examples).")
                return gai_extracted, 1.0
            except json.JSONDecodeError:
                return extracted, confidence

    return extracted, confidence


# ==============================================================================
# PHASE 3: CLASSIFICATION (FUNCTIONAL / NON-FUNCTIONAL)
# ==============================================================================
def phase3_classification(requirement_text):
    print("\n--- PHASE 3: Classification (with RAG) ---")
    if Config.USE_REAL_MODELS:
        print("⚙️ (Simulating) Real NoRBERT model is enabled.")
        classification, confidence = {"type": "Non-Functional", "subtype": "Performance"}, 0.80
        print(f"📉 (Simulated) Model confidence is low: {confidence:.2f}")
    else:
        print("⚙️ Using Keyword-based fallback for classification.")
        text_lower = requirement_text.lower()
        nfr_keywords = {
            "Performance": ["fast", "load", "response time", "seconds", "uptime", "latency", "throughput", "scalability", "ms", "efficient", "speed"],
            "Security": ["security", "authenticate", "encrypt", "multi-factor", "vulnerability", "rbac", "ssl/tls", "owasp", "firewall"],
            "Reliability": ["reliable", "uptime", "available", "99.9%", "fault tolerance", "robust", "downtime", "failover"],
            "Usability": ["user-friendly", "easy to use", "intuitive", "user experience", "accessible", "ui/ux"],
            "Maintainability": ["modular", "reusable", "clean code", "documented", "scalable architecture"]
        }
        found_subtype = None
        for subtype, keywords in nfr_keywords.items():
            if any(keyword in text_lower for keyword in keywords):
                found_subtype = subtype
                break
        if found_subtype:
            classification, confidence = {"type": "Non-Functional", "subtype": found_subtype}, 0.90
        else:
            classification, confidence = {"type": "Functional", "subtype": None}, 0.88
        print(f"📊 Keyword classification confidence: {confidence:.2f}")


    # 2. If Confidence Low -> USE RETRIEVAL AUGMENTED GENERATION
    if confidence < Config.CONFIDENCE_THRESHOLD and Config.USE_GAI_PHASE3:
        print(f"🤖 Engaging GAI with Dynamic Retrieval (RAG)...")

        # A. RETRIEVE 3 SIMILAR EXAMPLES
        dynamic_few_shot = get_dynamic_examples(requirement_text, mode="classification")

        # B. FORM AUGMENTED PROMPT
        prompt = f"""
        Classify the requirement as 'Functional' or 'Non-Functional'.
        If 'Non-Functional', provide a subtype.
        Respond with a JSON object.

        {dynamic_few_shot}

        --- YOUR TASK ---
        Requirement: "{requirement_text}"
        """

        # C. GENERATE Y
        json_response = api_manager.make_api_call(prompt, is_json_output=True)

        if json_response:
            try:
                gai_result = json.loads(json_response)
                print(f"✨ GAI Classification: {gai_result.get('type')} / {gai_result.get('subtype')}")
                return {"type": gai_result.get("type"), "subtype": gai_result.get("subtype")}, 1.0
            except json.JSONDecodeError:
                return classification, confidence

    return classification, confidence


# ==============================================================================
# PHASE 4: QUALITY VALIDATION (SMART CRITERIA)
# ==============================================================================
def calculate_semantic_agreement(requirement_text, classification):
    """
    Helper function to apply rule-based consistency checks.
    Returns a score (0.0-1.0) and a reason for failure.
    """
    is_nfr = classification.get('type') == 'Non-Functional'
    has_metric = bool(re.search(r'\d', requirement_text)) # Check if text contains any digit

    # RULE: Non-Functional Requirements should be measurable.
    if is_nfr and not has_metric:
        reason = "Inconsistency: Non-Functional Requirement lacks a measurable metric (e.g., a number)."
        return 0.6, reason # Penalize for lack of metric

    # RULE: Functional requirements are consistent by default for this demo.
    if not is_nfr:
        return 1.0, "Consistent: Functional requirement."

    return 1.0, "Consistent: Non-Functional Requirement has a measurable metric."

def phase4_quality_validation(requirement_text, classification_data, classification_confidence):
    """
    Performs a cross-perspective validation check based on the classification output.
    Uses a confidence gateway to trigger GAI for improvements.
    """
    print("\n--- PHASE 4: Cross-Perspective Quality Validation ---")

    # 1. Self-Validation & Cross-Perspective Check
    print("🔎 Performing consistency check between classification and requirement text...")
    initial_agreement_score, reason = calculate_semantic_agreement(requirement_text, classification_data)
    print(f"▶️ Check Result: {reason}")

    # 2. Evaluate Combined Confidence Score
    # Weighted average: 60% from the classifier's confidence, 40% from our rule-based check.
    combined_confidence_score = (0.6 * classification_confidence) + (0.4 * initial_agreement_score)
    print(f"📊 Classification Confidence: {classification_confidence:.2f}")
    print(f"📊 Semantic Agreement Score: {initial_agreement_score:.2f}")
    print(f"📈 Combined Validation Score: {combined_confidence_score:.2f}")

    final_text = requirement_text
    final_scores = {
        "classification_confidence": classification_confidence,
        "initial_semantic_agreement": initial_agreement_score,
        "combined_confidence_score": combined_confidence_score,
        "final_semantic_agreement": initial_agreement_score
    }

    # 3. Confidence Gateway
    if combined_confidence_score < Config.VALIDATION_THRESHOLD and Config.USE_GAI_PHASE4:
        print(f"🤖 Combined score is below threshold ({Config.VALIDATION_THRESHOLD}). Engaging GAI to resolve inconsistency...")
        prompt = f"""
        The following software requirement has a logical inconsistency.
        Your task is to rewrite the requirement to resolve the issue and make it a clear, high-quality statement. Do not invent new features.

        --- CONTEXT ---
        Original Requirement: "{requirement_text}"
        Detected Issue: {reason}

        --- EXAMPLE ---
        Original Requirement: "As a user, I want the system to be fast."
        Detected Issue: Inconsistency: Non-Functional Requirement lacks a measurable metric (e.g., a number).
        Improved Requirement: As a user, I want the system to respond to all actions within 2 seconds, so that I don't have to wait.

        --- YOUR TASK ---
        Original Requirement: "{requirement_text}"
        Detected Issue: {reason}
        Improved Requirement:
        """
        improved_text = api_manager.make_api_call(prompt)
        if improved_text and len(improved_text.strip()) > 10:
            print(f"✨ GAI Suggestion: {improved_text.strip()}")
            print("👤 Simulating user acceptance of GAI suggestion... ✅ Accepted.")
            final_text = improved_text.strip()

            # 4. Final Validation Pass
            print("🔎 Performing final validation pass on improved text...")
            final_agreement_score, final_reason = calculate_semantic_agreement(final_text, classification_data)
            final_scores['final_semantic_agreement'] = final_agreement_score
            print(f"▶️ Final Agreement Score: {final_agreement_score:.2f}. Reason: {final_reason}")
        else:
            print("❌ GAI suggestion failed. Using original text.")
    else:
        print(f"✅ Combined score is high or GAI is disabled. No improvement needed.")

    # The function signature expects three return values, but the code currently returns two.
    # The third value should be the overall quality score, which needs to be calculated.
    # TODO: Calculate the overall quality score based on final_scores.
    return final_text, final_scores, combined_confidence_score


# ==============================================================================
# PHASE 5: NEO4J GRAPH STORAGE & PHASE 6: TRACEABILITY DISCOVERY
# ==============================================================================
def phase5_neo4j_generation(req_id, requirement_text, extracted, classification, quality_scores):
    """
    Generates a single, atomic Cypher statement using a chained WITH clause
    pattern to ensure all nodes and relationships are created correctly.
    """
    print("\n--- PHASE 5: Neo4j Cypher Generation ---")

    clauses = []
    node_vars = ["req"]

    # Start with the Requirement node
    quality_score = quality_scores.get('combined_confidence_score', 0.0)
    safe_text = requirement_text.replace("'", "\\'")
    clauses.append(f"CREATE (req:Requirement {{ id: '{req_id}', text: '{safe_text}', quality_score: '{quality_score:.2f}' }})")

    # Chain other nodes with WITH clauses
    if extracted.get('actor'):
        clauses.append(f"WITH {', '.join(node_vars)}")
        safe_val = extracted['actor'].replace("'", "\\'")
        clauses.append(f"MERGE (actor_node:Actor {{ name: '{safe_val}' }})")
        node_vars.append("actor_node")

    if extracted.get('goal'):
        clauses.append(f"WITH {', '.join(node_vars)}")
        safe_val = extracted['goal'].replace("'", "\\'")
        clauses.append(f"MERGE (goal_node:Goal {{ text: '{safe_val}' }})")
        node_vars.append("goal_node")

    if extracted.get('rationale'):
        clauses.append(f"WITH {', '.join(node_vars)}")
        safe_val = extracted['rationale'].replace("'", "\\'")
        clauses.append(f"MERGE (rationale_node:Rationale {{ text: '{safe_val}' }})")
        node_vars.append("rationale_node")

    if classification.get("type"):
        clauses.append(f"WITH {', '.join(node_vars)}")
        class_label = "NFR" if classification['type'] == "Non-Functional" else "FR"
        subtype = classification.get("subtype", "General")
        clauses.append(f"MERGE (class_node:{class_label} {{ type: '{subtype}' }})")
        node_vars.append("class_node")

    # Final WITH before creating relationships
    clauses.append(f"WITH {', '.join(node_vars)}")

    # Build all relationships in the final CREATE clause
    rel_clauses = []
    if "actor_node" in node_vars: rel_clauses.append("(req)-[:HAS_ACTOR]->(actor_node)")
    if "goal_node" in node_vars: rel_clauses.append("(req)-[:HAS_GOAL]->(goal_node)")
    if "rationale_node" in node_vars: rel_clauses.append("(req)-[:HAS_RATIONALE]->(rationale_node)")
    if "class_node" in node_vars: rel_clauses.append("(req)-[:IS_CLASSIFIED_AS]->(class_node)")

    if rel_clauses:
        clauses.append(f"CREATE {', '.join(rel_clauses)}")

    return "\n".join(clauses)

def phase6_traceability_discovery(current_req_id, current_req_text, processed_requirements):
    print("\n--- PHASE 6: Traceability Discovery ---")
    if not processed_requirements or not Config.USE_GAI_PHASE6:
        print("Skipping traceability (no previous requirements or GAI is disabled).")
        return ""
    print("🤖 Engaging GAI to discover traceability links (Few-Shot)...")
    previous_req_list = "\n".join([f'- ID: {pid}, Text: "{ptext}"' for pid, ptext in processed_requirements.items()])
    prompt = f"""
    Analyze the 'Current Requirement' against the 'Previous Requirements'.
    Identify relationships (DEPENDS_ON, RELATES_TO, REFINES, CONFLICTS_WITH).
    Respond with a JSON list of objects, each with "target_id", "relationship_type", and "reason". If none, return an empty list.
    --- EXAMPLE ---
    Current Requirement:
    - ID: REQ-002, Text: "As a user, I want to pay with PayPal, so that I have more payment options."
    Previous Requirements:
    - ID: REQ-001, Text: "As a user, I want a checkout page, so that I can purchase items."
    [ {{"target_id": "REQ-001", "relationship_type": "DEPENDS_ON", "reason": "The ability to pay with PayPal depends on the existence of a checkout page."}} ]
    --- END EXAMPLE ---
    Current Requirement:
    - ID: {current_req_id}, Text: "{current_req_text}"
    Previous Requirements:
    {previous_req_list}
    """
    json_response = api_manager.make_api_call(prompt, is_json_output=True)
    if not json_response:
        print("❌ GAI traceability check failed.")
        return ""
    try:
        relationships = json.loads(json_response)
        if not relationships:
            print("✅ GAI found no significant relationships.")
            return ""
        print(f"✨ GAI discovered {len(relationships)} relationship(s):")
        cypher_queries = []
        for rel in relationships:
            clean_reason = rel.get("reason", "").replace("'", "\\'")
            query = f"""MATCH (a:Requirement {{ id: '{current_req_id}' }}), (b:Requirement {{ id: '{rel.get("target_id")}' }}) CREATE (a)-[:{rel.get("relationship_type")} {{ reason: '{clean_reason}' }}]->(b);"""
            cypher_queries.append(query)
            print(f"  - Found link: {current_req_id} -> {rel.get('target_id')} ({rel.get('relationship_type')})")
        return "\n".join(cypher_queries)
    except (json.JSONDecodeError, TypeError):
        print("❌ GAI did not return a valid list. Skipping traceability.")
        return ""

# ==============================================================================
# MAIN PROCESSING ORCHESTRATOR
# ==============================================================================
def process_requirement(requirement_text, processed_requirements):
    req_id = f"REQ-{str(uuid4())[:8]}"
    print(f"\n{'='*80}\nProcessing Requirement ID: {req_id}\nOriginal Text: \"{requirement_text}\"\n{'='*80}")

    p1_text = phase1_input_quality_check(requirement_text)
    p2_extracted, p2_conf = phase2_extraction(p1_text)
    print(f"▶️ Extracted Data: {p2_extracted}")
    p3_class, p3_conf = phase3_classification(p1_text)
    print(f"▶️ Classification: {p3_class}")
    p4_final_text, p4_scores, p4_overall = phase4_quality_validation(p1_text, p3_class, p3_conf) # Pass p3_conf as the third argument
    print(f"▶️ Final Requirement Text: {p4_final_text}")
    p5_cypher = phase5_neo4j_generation(req_id, p4_final_text, p2_extracted, p3_class, p4_scores) # Pass p4_scores here
    p6_cypher = phase6_traceability_discovery(req_id, p4_final_text, processed_requirements)

    result = {
        "id": req_id,
        "original_text": requirement_text,
        "final_requirement_text": p4_final_text,
        "neo4j_node_id": req_id,
        "extracted_actor": p2_extracted.get('actor'),
        "extracted_goal": p2_extracted.get('goal'),
        "extracted_rationale": p2_extracted.get('rationale'),
        "classification_type": p3_class.get('type'),
        "classification_subtype": p3_class.get('subtype'),
        "quality_scores": p4_scores,
        "cypher_queries": p5_cypher + ("\n" + p6_cypher if p6_cypher else "")
    }

    print(f"\n--- ✅ PROCESSING COMPLETE for {req_id} ---")
    processed_requirements[req_id] = p4_final_text
    return result

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.0 MB/s eta 0:00:00


In [8]:
test_requirements = ["The application's source code must be well-documented, with comments explaining complex algorithms, business logic, and non-obvious code sections."]

if not Config.GEMINI_API_KEY:
    print("\n\n" + "="*80 + "\n🚨 CRITICAL ERROR: Gemini API Key is not configured.\n" + "="*80 + "\n")
else:
    all_cypher_queries = [] # This list will contain ONLY pure Cypher
    processed_for_traceability = {}

    for req_text in tqdm(test_requirements, desc="Processing All Requirements"):
        result = process_requirement(req_text, processed_for_traceability)
        # --- THIS IS THE FIX ---
        # Append ONLY the executable Cypher script to the list
        all_cypher_queries.append(result['cypher_queries'])

    print(f"\n\n{'='*80}\n🎉 ALL REQUIREMENTS PROCESSED 🎉\n{'='*80}")
    print("\n\n--- COMPLETE NEO4J CYPHER SCRIPT (for inspection) ---")
    # We can add comments here for printing, but they are NOT in the list
    for i, script in enumerate(all_cypher_queries):
        print(f"\n-- Cypher for Requirement {i+1} --")
        print(script)

Processing All Requirements:   0%|          | 0/1 [00:00<?, ?it/s]


Processing Requirement ID: REQ-0c2d762e
Original Text: "The application's source code must be well-documented, with comments explaining complex algorithms, business logic, and non-obvious code sections."

--- PHASE 1: Input Quality Check ---
⚠️ Requirement is incomplete. Missing parts identified.
🤖 Engaging GAI to enhance the requirement (Few-Shot)...
✨ GAI Enhanced Requirement: As a developer, I want the application's source code to be well-documented with comments explaining complex logic, so that I can efficiently maintain, debug, and extend the codebase.

--- PHASE 2: Extraction (with RAG) ---
📊 Regex extraction confidence: 0.67
🤖 Engaging GAI with Dynamic Retrieval (RAG)...
✨ GAI extraction complete (Augmented with DB examples).
▶️ Extracted Data: {'actor': 'Developer', 'goal': 'Have well-documented source code with comments explaining complex logic', 'rationale': 'To efficiently maintain, debug, and extend the codebase'}

--- PHASE 3: Classification (with RAG) ---
⚙️ Using Keywo

In [9]:
# ==============================================================================
# CELL 2: LOAD DATA INTO NEO4J DATABASE
# ==============================================================================
#
# This cell connects to your Neo4j AuraDB instance, clears any old data,
# and executes the Cypher scripts generated by Cell 1. It correctly handles
# scripts that may contain multiple statements (e.g., for traceability links)
# by splitting them by the semicolon.

# ==============================================================================
# LIBRARIES & SETUP
# ==============================================================================
!pip install -q neo4j

from neo4j import GraphDatabase
from google.colab import userdata
import logging

# ==============================================================================
# DATABASE LOADER
# ==============================================================================
def load_data_to_neo4j(cypher_scripts):
    """Connects to Neo4j and executes a list of Cypher scripts."""
    print(f"\n{'='*80}\n🔄 Loading Data into Neo4j Database...\n{'='*80}")

    try:
        uri = userdata.get('NEO4J_URI')
        user = userdata.get('NEO4J_USERNAME')
        password = userdata.get('NEO4J_PASSWORD')
        if not all([uri, user, password]):
            raise ValueError("Neo4j credentials not found in Colab Secrets.")
    except Exception as e:
        logging.error(f"🚨 {e}")
        logging.error("👉 Please set NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD in Colab Secrets.")
        return

    # Establish a connection to the database
    driver = GraphDatabase.driver(uri, auth=(user, password))

    with driver.session() as session:
        # 1. Clear the database to ensure a fresh start
        print("🧹 Clearing old data from the database...")
        session.run("MATCH (n) DETACH DELETE n")

        # 2. Execute each generated script
        print(f"🚀 Loading {len(cypher_scripts)} new requirements...")
        for script in cypher_scripts:
            # The script might contain a main statement and an optional traceability
            # statement separated by a semicolon. We must run each one separately.
            for statement in script.split(';'):
                # Ensure the statement is not just whitespace
                if statement and statement.strip():
                    try:
                        session.run(statement)
                    except Exception as e:
                        print(f"--- FAILED CYPHER STATEMENT ---")
                        print(statement)
                        print(f"ERROR: {e}")
                        print("---------------------------------")

    driver.close()
    print("✅ Success! Data and relationships have been loaded into your Neo4j AuraDB instance.")

# ==============================================================================
# EXECUTION
# ==============================================================================
if 'all_cypher_queries' in locals() and all_cypher_queries:
    load_data_to_neo4j(all_cypher_queries)
else:
    logging.error("🚨 The `all_cypher_queries` variable was not found.")
    logging.error("👉 Please ensure you have successfully run Cell 1 before running this cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 17.5 MB/s eta 0:00:00

🔄 Loading Data into Neo4j Database...
🧹 Clearing old data from the database...
🚀 Loading 1 new requirements...
✅ Success! Data and relationships have been loaded into your Neo4j AuraDB instance.
